## Extract Face Images

In [1]:
import cv2
import os
import torch
from facenet_pytorch import MTCNN
from PIL import Image
import numpy as np
import torch_directml

# ============================================
# 🔧 CONFIGURATION
# ============================================
VIDEO_PATHS = {
    "krisis": r"data\2. Data Video\1. VIDEO KRISIS\1. VIDEO KRISIS",
    "tidak_krisis": r"data\2. Data Video\2. VIDEO TIDAK KRISIS"
}

OUTPUT_FACE_DIR = "data/final/face" 
FRAME_INTERVAL_SECONDS = 1.618  
CONFIDENCE_THRESHOLD = 0.85  
MIN_FACE_SIZE = 64           

# Format video valid
VIDEO_EXTENSIONS = (".mp4", ".mov", ".mkv", ".avi", ".wmv", ".flv", ".mpeg", ".mpg")

# Format gambar yang harus diabaikan
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ============================================
# 🧠 INITIALIZATION
# ============================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'# torch_directml.device()
print(f"🚀 Using device: {device}")

face_detector = MTCNN(keep_all=True, device=device)

for sub in ["krisis", "tidak_krisis"]:
    os.makedirs(os.path.join(OUTPUT_FACE_DIR, sub), exist_ok=True)

# ============================================
# 🎞️ FRAME EXTRACTION FUNCTION
# ============================================
def extract_frames(video_path, interval_sec=2):
    cap = cv2.VideoCapture(video_path)

    # Pastikan video benar-benar bisa dibuka
    if not cap.isOpened():
        print(f"❌ Tidak dapat membuka video: {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        print(f"⚠️ Tidak dapat membaca FPS untuk {video_path}")
        return []

    frame_interval = max(1, int(fps * interval_sec))
    frames = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frames.append(frame)
        frame_count += 1

    cap.release()
    return frames

# ============================================
# 🙂 FACE DETECTION FUNCTION
# ============================================
def detect_and_save_faces(frame, base_name, output_dir):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    boxes, probs = face_detector.detect(img_pil)

    if boxes is None:
        return 0

    valid_faces = 0
    for i, (box, prob) in enumerate(zip(boxes, probs)):
        if prob is None or prob < CONFIDENCE_THRESHOLD:
            continue  

        x1, y1, x2, y2 = [int(b) for b in box]
        w, h = x2 - x1, y2 - y1

        if w < MIN_FACE_SIZE or h < MIN_FACE_SIZE:
            continue
        aspect_ratio = w / h
        if aspect_ratio < 0.7 or aspect_ratio > 1.3:
            continue

        face_crop = frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue

        face_crop = cv2.resize(face_crop, (224, 224))
        out_path = os.path.join(output_dir, f"{base_name}_face_{i}.jpg")
        cv2.imwrite(out_path, face_crop)
        valid_faces += 1

    return valid_faces

# ============================================
# 🔁 MAIN PIPELINE
# ============================================
def process_videos(label, video_dir):
    all_files = os.listdir(video_dir)

    # FILTER:
    video_files = [
        f for f in all_files
        if f.lower().endswith(VIDEO_EXTENSIONS) and not f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    print(f"\n🧠 Processing category: {label} ({len(video_files)} video files found)\n")

    total_faces = 0

    for idx, video_file in enumerate(video_files):
        video_path = os.path.join(video_dir, video_file)
        print(f"[{idx+1}/{len(video_files)}] 🎬 Processing: {video_file}")

        frames = extract_frames(video_path, interval_sec=FRAME_INTERVAL_SECONDS)
        if not frames:
            print(f"⚠️ Skipping {video_file}, no frames extracted.")
            continue

        face_dir = os.path.join(OUTPUT_FACE_DIR, label)
        face_count_video = 0

        for i, frame in enumerate(frames):
            base_name = f"{os.path.splitext(video_file)[0]}_frame_{i:04d}"
            face_count_video += detect_and_save_faces(frame, base_name, face_dir)

        total_faces += face_count_video
        print(f"✅ Done: {video_file} — {face_count_video} valid faces saved\n")

    print(f"📸 Total wajah valid kategori '{label}': {total_faces}\n")

# ============================================
# 🚀 RUN
# ============================================
if __name__ == "__main__":
    for label, path in VIDEO_PATHS.items():
        process_videos(label, path)
    print("\n🎉 Semua video selesai diproses dengan filter wajah valid!")


d:\Projects\Robot Pencegah Bunuh Diri\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Using device: cpu

🧠 Processing category: krisis (870 video files found)

[1/870] 🎬 Processing: K1.mp4
✅ Done: K1.mp4 — 3 valid faces saved

[2/870] 🎬 Processing: k10.mp4
✅ Done: k10.mp4 — 2 valid faces saved

[3/870] 🎬 Processing: K100.mp4
✅ Done: K100.mp4 — 4 valid faces saved

[4/870] 🎬 Processing: K1000.mp4
✅ Done: K1000.mp4 — 2 valid faces saved

[5/870] 🎬 Processing: K1001.mp4
✅ Done: K1001.mp4 — 1 valid faces saved

[6/870] 🎬 Processing: K1002.mp4
✅ Done: K1002.mp4 — 0 valid faces saved

[7/870] 🎬 Processing: K1003.mp4
✅ Done: K1003.mp4 — 0 valid faces saved

[8/870] 🎬 Processing: K1004.mp4
✅ Done: K1004.mp4 — 0 valid faces saved

[9/870] 🎬 Processing: K1005.mp4
✅ Done: K1005.mp4 — 0 valid faces saved

[10/870] 🎬 Processing: K1006.mp4
✅ Done: K1006.mp4 — 0 valid faces saved

[11/870] 🎬 Processing: K1007.mp4
✅ Done: K1007.mp4 — 0 valid faces saved

[12/870] 🎬 Processing: K1008.mp4
✅ Done: K1008.mp4 — 0 valid faces saved

[13/870] 🎬 Processing: K1009.mp4
✅ Done: K1009.mp4 — 0 va

## Extract Pose/Body Images

In [1]:
import cv2
import os
import mediapipe as mp
import numpy as np

# ============================================
# 🔧 CONFIGURATION
# ============================================
VIDEO_PATHS = {
    "krisis": r"data\2. Data Video\1. VIDEO KRISIS\1. VIDEO KRISIS",
    "tidak_krisis": r"data\2. Data Video\2. VIDEO TIDAK KRISIS"
}

OUTPUT_POSE_DIR = "data/final/pose"
FRAME_INTERVAL_SECONDS = 1.5

VIDEO_EXTENSIONS = (".mp4", ".mov", ".mkv", ".avi", ".wmv", ".flv", ".mpeg", ".mpg")
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ============================================
# 🧠 INITIALIZATION
# ============================================
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# High quality drawing settings
skeleton_style_landmark = mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=3, circle_radius=3)
skeleton_style_connection = mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)

pose_detector = mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5,
    model_complexity=2
)

# Create output folders
for sub in ["krisis", "tidak_krisis"]:
    os.makedirs(os.path.join(OUTPUT_POSE_DIR, sub), exist_ok=True)

# ============================================
# 🎞️ FRAME EXTRACTION
# ============================================
def extract_frames(video_path, interval_sec=2):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ Cannot open video: {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        print(f"⚠️ Cannot read FPS for {video_path}")
        return []

    frame_interval = max(1, int(fps * interval_sec))
    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_interval == 0:
            frames.append(frame)

        frame_id += 1

    cap.release()
    return frames

# ============================================
# 🧍 DRAW SKELETON ON IMAGE
# ============================================
def draw_skeleton(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose_detector.process(rgb)

    if not result.pose_landmarks:
        return None

    annotated = frame.copy()
    mp_drawing.draw_landmarks(
        annotated,
        result.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=skeleton_style_landmark,
        connection_drawing_spec=skeleton_style_connection,
    )
    return annotated

# ============================================
# ✂️ SAVE 224x224 IMAGE
# ============================================
def save_pose_image(frame, base_name, output_dir):
    skeleton_img = draw_skeleton(frame)
    if skeleton_img is None:
        return False

    img224 = cv2.resize(skeleton_img, (224, 224))

    output_path = os.path.join(output_dir, f"{base_name}_pose.jpg")
    cv2.imwrite(output_path, img224)

    return True

# ============================================
# 🔁 MAIN PIPELINE
# ============================================
def process_videos_pose(label, video_dir):
    all_files = os.listdir(video_dir)

    video_files = [
        f for f in all_files
        if f.lower().endswith(VIDEO_EXTENSIONS) and not f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    print(f"\n🧠 Processing SKELETON + IMAGE for: {label} ({len(video_files)} videos)\n")

    total_saved = 0

    for idx, video_file in enumerate(video_files):
        video_path = os.path.join(video_dir, video_file)
        print(f"[{idx+1}/{len(video_files)}] 🎬 Processing: {video_file}")

        frames = extract_frames(video_path, interval_sec=FRAME_INTERVAL_SECONDS)
        if not frames:
            print(f"⚠️ Skipping {video_file} — no frames extracted")
            continue

        output_dir = os.path.join(OUTPUT_POSE_DIR, label)
        saved_count = 0

        for i, frame in enumerate(frames):
            base_name = f"{os.path.splitext(video_file)[0]}_frame_{i:04d}"

            if save_pose_image(frame, base_name, output_dir):
                saved_count += 1

        total_saved += saved_count
        print(f"✅ Done: {video_file} — {saved_count} skeleton images saved\n")

    print(f"🧍 TOTAL skeleton images for '{label}': {total_saved}")

# ============================================
# 🚀 RUN
# ============================================
if __name__ == "__main__":
    for label, path in VIDEO_PATHS.items():
        process_videos_pose(label, path)

    print("\n🎉 ALL VIDEOS PROCESSED — SKELETON DATASET READY!")


🧠 Processing SKELETON + IMAGE for: krisis (870 videos)

[1/870] 🎬 Processing: K1.mp4
✅ Done: K1.mp4 — 4 skeleton images saved

[2/870] 🎬 Processing: k10.mp4
✅ Done: k10.mp4 — 3 skeleton images saved

[3/870] 🎬 Processing: K100.mp4
✅ Done: K100.mp4 — 4 skeleton images saved

[4/870] 🎬 Processing: K1000.mp4
✅ Done: K1000.mp4 — 3 skeleton images saved

[5/870] 🎬 Processing: K1001.mp4
✅ Done: K1001.mp4 — 2 skeleton images saved

[6/870] 🎬 Processing: K1002.mp4
✅ Done: K1002.mp4 — 3 skeleton images saved

[7/870] 🎬 Processing: K1003.mp4
✅ Done: K1003.mp4 — 3 skeleton images saved

[8/870] 🎬 Processing: K1004.mp4
✅ Done: K1004.mp4 — 3 skeleton images saved

[9/870] 🎬 Processing: K1005.mp4
✅ Done: K1005.mp4 — 2 skeleton images saved

[10/870] 🎬 Processing: K1006.mp4
✅ Done: K1006.mp4 — 3 skeleton images saved

[11/870] 🎬 Processing: K1007.mp4
✅ Done: K1007.mp4 — 2 skeleton images saved

[12/870] 🎬 Processing: K1008.mp4
✅ Done: K1008.mp4 — 4 skeleton images saved

[13/870] 🎬 Processing: K1009